In [ ]:
import os
import shutil
from google.colab import drive

# 1. 掛載雲端硬碟
drive.mount('/content/drive')

# 2. 設定來源路徑與雲端硬碟目標路徑
local_folder = '/content/TXO_Daily_Results'
drive_target_folder = '/content/drive/MyDrive/TXO_Daily_Results_2021'

# 3. 在雲端硬碟建立目標資料夾
if not os.path.exists(drive_target_folder):
    os.makedirs(drive_target_folder)
    print(f"📁 已在雲端硬碟建立資料夾: {drive_target_folder}")

# 4. 開始逐一複製檔案
files_to_copy = os.listdir(local_folder)

if not files_to_copy:
    print("❌ 錯誤：Colab 暫存區內沒有任何可轉移的檔案。請檢查先前的篩選步驟。")
else:
    print(f"🚀 開始將 {len(files_to_copy)} 個檔案上傳至雲端硬碟...")

    count = 0
    for file_name in files_to_copy:
        source = os.path.join(local_folder, file_name)
        destination = os.path.join(drive_target_folder, file_name)

        # 執行複製
        shutil.copy(source, destination)
        count += 1
        if count % 10 == 0:  # 每 10 個檔案顯示一次進度
            print(f"已上傳 {count} 個檔案...")

    print("-" * 30)
    print(f"✨ 上傳完成！共計 {count} 個檔案。")
    print(f"📍 你可以在雲端硬碟的 [TXO_Daily_Results_2021] 資料夾中找到它們。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 已在雲端硬碟建立資料夾: /content/drive/MyDrive/TXO_Daily_Results_2021
🚀 開始將 244 個檔案上傳至雲端硬碟...
已上傳 10 個檔案...
已上傳 20 個檔案...
已上傳 30 個檔案...
已上傳 40 個檔案...
已上傳 50 個檔案...
已上傳 60 個檔案...
已上傳 70 個檔案...
已上傳 80 個檔案...
已上傳 90 個檔案...
已上傳 100 個檔案...
已上傳 110 個檔案...
已上傳 120 個檔案...
已上傳 130 個檔案...
已上傳 140 個檔案...
已上傳 150 個檔案...
已上傳 160 個檔案...
已上傳 170 個檔案...
已上傳 180 個檔案...
已上傳 190 個檔案...
已上傳 200 個檔案...
已上傳 210 個檔案...
已上傳 220 個檔案...
已上傳 230 個檔案...
已上傳 240 個檔案...
------------------------------
✨ 上傳完成！共計 244 個檔案。
📍 你可以在雲端硬碟的 [TXO_Daily_Results_2021] 資料夾中找到它們。


In [3]:
# ==========================================
# 第一部分 : 環境準備與套件安裝 (組員初次執行請等待約 10 秒)
# ==========================================
!pip install py_vollib_vectorized -q


import pandas as pd
import numpy as np
import zipfile
import py_vollib_vectorized
from google.colab import drive
from google.colab import files
import os
import io
import warnings


# 忽略特定計算警告
warnings.filterstrict = False
warnings.filterwarnings('ignore', category=UserWarning, module='py_vollib_vectorized')


# 掛載 Google Drive (會跳出授權視窗，請組員允許)
drive.mount('/content/drive')




# @markdown 請在右側填寫對應的檔案路徑：
索引檔路徑 = '/content/drive/MyDrive/Index_411336064_2021.csv' # @param {type:"string"}
逐筆資料壓縮檔路徑 = '/content/drive/MyDrive/Option_2021.zip' # @param {type:"string"}


# @markdown 如果壓縮檔點開後，裡面還包了一層資料夾，請填寫資料夾名稱（例如 `Option_2023`）。若沒有請留空：
ZIP內部資料夾名稱 = 'Option_2021' # @param {type:"string"}


# @markdown 請為本次分析輸出的檔案命名：
輸出CSV檔名 = 'TXO_Analysis_Result.csv' # @param {type:"string"}


# 自動處理路徑與名稱變數 (供底層程式使用)
INDEX_FILE_PATH = 索引檔路徑.strip()
OPTIONS_DATA_PATH = 逐筆資料壓縮檔路徑.strip()
OUTPUT_CSV_PATH = f'/content/{輸出CSV檔名.strip()}'


# 自動補齊資料夾斜線
ZIP_FOLDER_PREFIX = ZIP內部資料夾名稱.strip()
if ZIP_FOLDER_PREFIX and not ZIP_FOLDER_PREFIX.endswith('/'):
    ZIP_FOLDER_PREFIX += '/'


# ==========================================
# 第三部分 : 核心運算模組 (修正版)
# ==========================================
def process_options_data():
    print("⏳ [1/3] 正在讀取並解析索引檔...")
    if not os.path.exists(INDEX_FILE_PATH):
        print(f"❌ 嚴重錯誤：找不到索引檔，請檢查路徑是否正確 [{INDEX_FILE_PATH}]")
        return None

    try:
        # --- 核心修正點：嘗試多種編碼，優先處理包含 BOM 的 UTF-8 ---
        if INDEX_FILE_PATH.endswith('.xlsx'):
            index_df = pd.read_excel(INDEX_FILE_PATH)
        else:
            try:
                # 使用 utf-8-sig 可以自動處理檔案開頭的 \ufeff
                index_df = pd.read_csv(INDEX_FILE_PATH, encoding='utf-8-sig')
            except:
                # 如果 utf-8 不行，再退回 cp950 (大五碼)
                index_df = pd.read_csv(INDEX_FILE_PATH, encoding='cp950')

        # 強制清理欄位名稱的前後空白
        index_df.columns = index_df.columns.str.strip()

    except Exception as e:
        print(f"❌ 無法讀取索引檔: {e}"); return None

    # 確保索引檔具備必要欄位
    required_cols = ['Date', 'File', 'S0', 'Maturity', 'Rf']
    missing_cols = [col for col in required_cols if col not in index_df.columns]

    if missing_cols:
        print(f"❌ 索引檔缺少必要欄位: {missing_cols}")
        print(f"💡 目前偵測到的欄位有: {list(index_df.columns)}")
        return None


# ==========================================
# 第四部分 : 執行與結果匯出
# ==========================================
analyzed_df = process_options_data()


if analyzed_df is not None and not analyzed_df.empty:
    print(f"\n🎉 分析全部完成！正在產生並下載 {輸出CSV檔名} ...")
    analyzed_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
    files.download(OUTPUT_CSV_PATH)
    display(analyzed_df.head())
else:
    print('\n⚠️ 未能產出資料。請檢查上方的錯誤訊息。')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ [1/3] 正在讀取並解析索引檔...

⚠️ 未能產出資料。請檢查上方的錯誤訊息。


In [16]:
# ==========================================
# 第一部分 : 環境準備與套件安裝 (組員初次執行請等待約 10 秒)
# ==========================================
!pip install py_vollib_vectorized -q


import pandas as pd
import numpy as np
import zipfile
import py_vollib_vectorized
from google.colab import drive
from google.colab import files
import os
import io
import warnings


# 忽略特定計算警告
warnings.filterstrict = False
warnings.filterwarnings('ignore', category=UserWarning, module='py_vollib_vectorized')


# 掛載 Google Drive (會跳出授權視窗，請組員允許)
drive.mount('/content/drive')




# @markdown 請在右側填寫對應的檔案路徑：
索引檔路徑 = '/content/drive/MyDrive/Index_411336064_2021.csv' # @param {type:"string"}
逐筆資料壓縮檔路徑 = '/content/drive/MyDrive/Option_2021.zip' # @param {type:"string"}


# @markdown 如果壓縮檔點開後，裡面還包了一層資料夾，請填寫資料夾名稱（例如 `Option_2023`）。若沒有請留空：
ZIP內部資料夾名稱 = 'Option_2021' # @param {type:"string"}


# @markdown 請為本次分析輸出的檔案命名：
輸出CSV檔名 = 'TXO_Analysis_Result.csv' # @param {type:"string"}


# 自動處理路徑與名稱變數 (供底層程式使用)
INDEX_FILE_PATH = 索引檔路徑.strip()
OPTIONS_DATA_PATH = 逐筆資料壓縮檔路徑.strip()
OUTPUT_CSV_PATH = f'/content/{輸出CSV檔名.strip()}'


# 自動補齊資料夾斜線
ZIP_FOLDER_PREFIX = ZIP內部資料夾名稱.strip()
if ZIP_FOLDER_PREFIX and not ZIP_FOLDER_PREFIX.endswith('/'):
    ZIP_FOLDER_PREFIX += '/'


# ==========================================
# 第三部分 : 核心運算模組 (完整修復版)
# ==========================================
import re

def process_options_data():
    print("⏳ [1/3] 正在讀取並解析索引檔...")
    try:
        index_df = pd.read_csv(INDEX_FILE_PATH, encoding='utf-8-sig')
        index_df.columns = index_df.columns.str.strip()
    except Exception as e:
        print(f"❌ 無法讀取索引檔: {e}"); return None

    index_df['Date'] = pd.to_datetime(index_df['Date'])
    results = []

    print(f"📦 [2/3] 正在讀取主壓縮檔並建立數字索引...")
    try:
        z_main = zipfile.ZipFile(OPTIONS_DATA_PATH, 'r')
        all_names = [n for n in z_main.namelist() if not n.endswith('/')]

        # 建立數字指紋地圖：從路徑中提取所有數字
        # 例如 'Option/OptionsDaily_2021_01_04.csv' -> '20210104'
        fingerprint_map = {}
        for path in all_names:
            nums = "".join(re.findall(r'\d+', os.path.basename(path)))
            if nums: fingerprint_map[nums] = path

        print(f"💡 壓縮檔掃描完成，已建立 {len(fingerprint_map)} 組日期指紋。")
    except Exception as e:
        print(f"❌ 無法開啟壓縮檔: {e}"); return None

    print(f"🚀 [3/3] 開始強制比對數字指紋...")

    for idx, row in index_df.iterrows():
        # 從索引檔的檔名欄位提取數字
        target_nums = "".join(re.findall(r'\d+', str(row['File'])))
        actual_path = fingerprint_map.get(target_nums)

        if not actual_path:
            continue

        try:
            with z_main.open(actual_path) as f:
                # 這裡強制讀取，若 cp950 失敗則嘗試 utf-8
                try:
                    df = pd.read_csv(f, encoding='cp950', low_memory=False)
                except:
                    f.seek(0)
                    df = pd.read_csv(f, encoding='utf-8', low_memory=False)

            df.columns = df.columns.str.strip()

            # --- 精準篩選條件 ---
            # 1. 商品代號
            c_id = next((c for c in df.columns if '商品' in c), None)
            if c_id:
                df = df[df[c_id].astype(str).str.contains('TXO', na=False)].copy()

            # 2. 成交數量 > 30 (處理逗號與空白)
            c_v = next((c for c in df.columns if '成交數量' in c), None)
            if c_v:
                df[c_v] = pd.to_numeric(df[c_v].astype(str).str.replace(',', '').str.strip(), errors='coerce')
                df = df[df[c_v] > 30].copy()

            if df.empty: continue

            # 3. 計算 IV
            s0, rf, t_year = row['S0'], row['Rf'], max(row['Maturity'], 0.5)/365.0
            c_cp = next((c for c in df.columns if '買賣' in c), None)
            c_k = next((c for c in df.columns if '履約' in c), None)
            c_p = next((c for c in df.columns if '成交價格' in c), None)

            if not all([c_cp, c_k, c_p]): continue

            # 標籤化買賣權
            flags = df[c_cp].astype(str).apply(lambda x: 'c' if any(k in x for k in ['買', 'C', 'c']) else 'p')

            # 執行向量化計算
            df['IV'] = py_vollib_vectorized.vectorized_implied_volatility(
                price=pd.to_numeric(df[c_p], errors='coerce'),
                S=s0, K=pd.to_numeric(df[c_k], errors='coerce'),
                t=t_year, r=rf, flag=flags.values, return_as='numpy'
            )

            df = df.dropna(subset=['IV'])
            calls, puts = df[flags == 'c'], df[flags == 'p']

            if not df.empty:
                cv, pv = calls[c_v].sum(), puts[c_v].sum()
                results.append({
                    'Date': row['Date'].strftime('%Y-%m-%d'),
                    'Call_IV_Mean': calls['IV'].mean(), 'Call_IV_Std': calls['IV'].std(),
                    'Put_IV_Mean': puts['IV'].mean(), 'Put_IV_Std': puts['IV'].std(),
                    'Call_Vol': cv, 'Put_Vol': pv,
                    'PCR': pv / cv if cv > 0 else 0
                })
                if len(results) % 20 == 0:
                    print(f"  ✅ 成功計算至: {row['Date'].date()}")

        except Exception:
            continue

    z_main.close()
    return pd.DataFrame(results) if results else None


# ==========================================
# 第四部分 : 執行與結果匯出
# ==========================================
analyzed_df = process_options_data()


if analyzed_df is not None and not analyzed_df.empty:
    print(f"\n🎉 分析全部完成！正在產生並下載 {輸出CSV檔名} ...")
    analyzed_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
    files.download(OUTPUT_CSV_PATH)
    display(analyzed_df.head())
else:
    print('\n⚠️ 未能產出資料。請檢查上方的錯誤訊息。')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ [1/3] 正在讀取並解析索引檔...
📦 [2/3] 正在讀取主壓縮檔並建立數字索引...
💡 壓縮檔掃描完成，已建立 244 組日期指紋。
🚀 [3/3] 開始強制比對數字指紋...


/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipyke

  ✅ 成功計算至: 2021-01-29


/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipyke

  ✅ 成功計算至: 2021-03-10


...

/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']
/tmp/ipykernel_938/2685476653.py:140: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  calls, puts = df[flags == 'c'], df[flags == 'p']



🎉 分析全部完成！正在產生並下載 TXO_Analysis_Result.csv ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Date,Call_IV_Mean,Call_IV_Std,Put_IV_Mean,Put_IV_Std,Call_Vol,Put_Vol,PCR
0,2021-01-04,0.075236,0.047777,0.115625,0.073849,66137.0,113062.0,1.709512
1,2021-01-05,0.050953,0.043874,0.087726,0.061587,76038.0,159164.0,2.093217
2,2021-01-06,0.070426,0.057613,0.069893,0.098780,234216.0,293440.0,1.252861
3,2021-01-07,0.114747,0.042927,0.192914,0.069221,81327.0,97322.0,1.196675
4,2021-01-08,0.124079,0.045924,0.193241,0.086784,93158.0,113433.0,1.217641


In [17]:
import shutil
import os

# 1. 設定原始檔案路徑（剛剛產出的檔案）
source_file = '/content/TXO_Analysis_Result.csv'

# 2. 設定目標路徑（Google Drive 的根目錄）
# 你可以修改這裡的名稱來決定存放在 Drive 的哪個資料夾
destination_path = '/content/drive/MyDrive/TXO_Analysis_Result.csv'

# 3. 執行傳送
if os.path.exists(source_file):
    try:
        shutil.copy(source_file, destination_path)
        print(f"✅ 檔案上傳成功！")
        print(f"路徑：Google Drive 根目錄下的 {os.path.basename(destination_path)}")
    except Exception as e:
        print(f"❌ 上傳失敗：{e}")
else:
    print(f"❌ 找不到原始檔案：{source_file}，請確認檔名是否正確。")

✅ 檔案上傳成功！
路徑：Google Drive 根目錄下的 TXO_Analysis_Result.csv
